# Redes Convolucionales — Qué aprende una CNN
### CNNs con Keras

## Intro

En el notebook anterior construiste una MLP para clasificación binaria. Esta red trataba todas las features como **independientes entre sí**. Cuando calculas `Dense(16)`, cada neurona ve todas las 30 features del dataset por igual.

Eso funciona con datos tabulares. Pero imagina que la entrada es una imagen de 28×28 píxeles (784 features). ¿Tiene sentido que una neurona conecte el píxel de la esquina superior izquierda con el de la esquina inferior derecha?

Un '3' escrito a mano sigue siendo un '3' aunque esté desplazado dos píxeles a la derecha. Una red densa no puede aprender esa invarianza a la traslación — trataría el '3' desplazado como si fuera un dígito completamente diferente.

**Las CNN** resuelven exactamente esto: en lugar de conectar cada neurona a todos los píxeles, usan **filtros locales** que se deslizan por la imagen. Aprenden a detectar bordes, esquinas, texturas... independientemente de dónde estén en la imagen.Pero más importante que aprender a usarlas es entender **qué aprenden** 



---
## Bloque 0 — Setup

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.ndimage as ndi

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.manifold import TSNE

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print("✅ Setup completo")

---
## Bloque 1 — El problema con las redes densas en imágenes

### MNIST: el dataset de referencia

MNIST es una colección de 70.000 dígitos escritos a mano (0-9), cada uno en una imagen de 28×28 píxeles en escala de grises. Es el "Hello World" del deep learning con imágenes.

Primero vamos a ver qué puede hacer una MLP normal. Spoiler: funciona bastante bien. Eso es parte del argumento — MNIST es tan fácil que casi cualquier cosa funciona. El problema real aparece con imágenes más complejas.

In [ ]:
# Cargamos MNIST — se descarga automáticamente la primera vez (~11MB)
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

print(f"Train: {X_train_raw.shape} — {X_train_raw.shape[0]} imágenes de {X_train_raw.shape[1]}×{X_train_raw.shape[2]} píxeles")
print(f"Test:  {X_test_raw.shape}")
print(f"Valores de píxel: [{X_train_raw.min()}, {X_train_raw.max()}]")

# Visualizamos algunos ejemplos
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_raw[i], cmap='gray')
    ax.set_title(str(y_train[i]), fontsize=10)
    ax.axis('off')
plt.suptitle('Primeros 20 ejemplos de MNIST', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Preprocesamiento para MLP: aplanamos (28x28 → 784) y normalizamos
X_train_flat = X_train_raw.reshape(-1, 784).astype('float32') / 255.0
X_test_flat  = X_test_raw.reshape(-1, 784).astype('float32') / 255.0

# MLP baseline — igual que el notebook anterior pero con 10 clases
mlp_mnist = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')  # 10 clases → softmax, no sigmoid
], name='MLP_MNIST')

mlp_mnist.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print("Arquitectura MLP:")
mlp_mnist.summary()

> Usamos `sparse_categorical_crossentropy` en lugar de `binary_crossentropy` porque ahora tenemos 10 clases, no 2. Y `softmax` en la última capa en lugar de sigmoid — softmax garantiza que las probabilidades de las 10 clases sumen exactamente 1.

In [ ]:
hist_mlp = mlp_mnist.fit(
    X_train_flat, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    verbose=0
)

loss_mlp, acc_mlp = mlp_mnist.evaluate(X_test_flat, y_test, verbose=0)
print(f"MLP — Accuracy en test: {acc_mlp:.4f} ({acc_mlp*100:.1f}%)")

In [ ]:
# El problema que tiene la MLP: sensibilidad a la posición
# Tomamos un '3' y lo desplazamos

# Buscamos un '3' limpio
idx_3 = np.where(y_train == 3)[0][0]
imagen_original = X_train_raw[idx_3].astype('float32') / 255.0

# Desplazamos la imagen 4 píxeles a la derecha
imagen_desplazada = np.roll(imagen_original, shift=4, axis=1)

# Predicciones del MLP para ambas
pred_orig = mlp_mnist.predict(imagen_original.reshape(1, 784), verbose=0)[0]
pred_desp = mlp_mnist.predict(imagen_desplazada.reshape(1, 784), verbose=0)[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(imagen_original, cmap='gray')
axes[0].set_title(f'Original\nPredicción: {pred_orig.argmax()} ({pred_orig.max():.0%})', fontsize=11)
axes[0].axis('off')

axes[1].imshow(imagen_desplazada, cmap='gray')
axes[1].set_title(f'Desplazada 4px\nPredicción: {pred_desp.argmax()} ({pred_desp.max():.0%})', fontsize=11)
axes[1].axis('off')

# Diferencia como la ve la MLP (a nivel de vector aplanado)
diff = np.abs(imagen_original - imagen_desplazada)
axes[2].imshow(diff, cmap='Reds')
axes[2].set_title(f'Píxeles diferentes\n{(diff > 0).sum()} de 784 píxeles cambiaron', fontsize=11)
axes[2].axis('off')

plt.suptitle('Para la MLP, estos son dos entradas distintas', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Probabilidades para la imagen ORIGINAL:")
for i, p in enumerate(pred_orig):
    bar = '█' * int(p * 30)
    print(f"  {i}: {bar} {p:.3f}")

print("\nProbabilidades para la imagen DESPLAZADA:")
for i, p in enumerate(pred_desp):
    bar = '█' * int(p * 30)
    print(f"  {i}: {bar} {p:.3f}")

---
## Bloque 2 — ¿Qué es un filtro convolucional?

Antes de usar Keras, vamos a entender qué hace una convolución a mano.

Un **filtro** (o kernel) es una pequeña matriz de pesos, normalmente 3×3 o 5×5. Se desliza sobre la imagen y en cada posición calcula el **producto punto** entre el filtro y el parche de imagen que cubre. El resultado es un **feature map**: una nueva imagen donde cada píxel indica "cuánto se parece este parche al filtro".

La clave: **los filtros no se diseñan a mano**. La red los aprende durante el entrenamiento. Pero para entender qué aprende, primero veamos filtros clásicos diseñados a mano.

In [ ]:
# Tomamos una imagen de MNIST para experimentar
img = X_train_raw[0].astype('float32')  # Un '5'

# Filtros clásicos diseñados a mano
filtros = {
    'Bordes horizontales\n(Sobel H)': np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1]
    ], dtype=float),

    'Bordes verticales\n(Sobel V)': np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=float),

    'Bordes diagonales': np.array([
        [ 2,  1,  0],
        [ 1,  0, -1],
        [ 0, -1, -2]
    ], dtype=float),

    'Nitidez\n(sharpening)': np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0]
    ], dtype=float),
}

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Fila superior: filtros como imágenes
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Imagen original', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')

for i, (nombre, kernel) in enumerate(filtros.items()):
    axes[0, i+1].imshow(kernel, cmap='RdBu_r', vmin=-2, vmax=2)
    axes[0, i+1].set_title(f'Filtro:\n{nombre}', fontsize=9)
    for r in range(3):
        for c in range(3):
            axes[0, i+1].text(c, r, f'{kernel[r,c]:.0f}',
                              ha='center', va='center', fontsize=11, fontweight='bold')
    axes[0, i+1].axis('off')

# Fila inferior: feature maps resultantes
axes[1, 0].axis('off')  # vacío

for i, (nombre, kernel) in enumerate(filtros.items()):
    fmap = ndi.convolve(img, kernel)
    fmap_clip = np.clip(fmap, 0, None)  # ReLU implícito: nos quedamos con activaciones positivas
    axes[1, i+1].imshow(fmap_clip, cmap='hot')
    axes[1, i+1].set_title('Feature map resultante', fontsize=9)
    axes[1, i+1].axis('off')

plt.suptitle('Imagen → Filtro → Feature map\nCada filtro detecta un tipo de patrón diferente',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Cada filtro produce un **feature map** distinto — una imagen que resalta el tipo de patrón que detecta ese filtro. El filtro de bordes horizontales solo se activa donde hay un cambio de intensidad de arriba a abajo. El de verticales, donde hay un cambio de izquierda a derecha.

**La intuición fundamental**: en lugar de darle a la red los 784 píxeles en bruto, la convolución transforma la imagen en un mapa de "dónde están los patrones". Y esos patrones se van haciendo más abstractos en capas más profundas.

Lo que hace Keras en `Conv2D(32, kernel_size=3)` es exactamente esto, pero con 32 filtros aprendidos en lugar de 4 diseñados a mano. La red decide qué patrones buscar.

### ¿Qué es MaxPooling?

Después de cada capa convolucional suele ir un **MaxPooling**: divide el feature map en bloques de 2×2 y se queda con el valor máximo de cada bloque. Dos efectos:
1. **Reduce el tamaño** a la mitad (28×28 → 14×14) — menos parámetros
2. **Invarianza local**: si el patrón se desplaza 1 píxel, el máximo del bloque no cambia

In [ ]:
# Visualizamos MaxPooling sobre el feature map de bordes horizontales
kernel_h = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=float)
fmap = np.clip(ndi.convolve(img, kernel_h), 0, None)

# MaxPool 2x2 manual
h, w = fmap.shape
pooled = fmap[:h//2*2, :w//2*2].reshape(h//2, 2, w//2, 2).max(axis=(1,3))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Original\n{img.shape[0]}×{img.shape[1]}', fontsize=11)
axes[0].axis('off')

axes[1].imshow(fmap, cmap='hot')
axes[1].set_title(f'Tras Conv2D (bordes H)\n{fmap.shape[0]}×{fmap.shape[1]}', fontsize=11)
axes[1].axis('off')

axes[2].imshow(pooled, cmap='hot')
axes[2].set_title(f'Tras MaxPool 2×2\n{pooled.shape[0]}×{pooled.shape[1]} ← mitad del tamaño', fontsize=11)
axes[2].axis('off')

plt.suptitle('MaxPooling: reduce tamaño, conserva los patrones más fuertes', fontsize=12)
plt.tight_layout()
plt.show()

---
## Bloque 3 — La CNN sobre MNIST

Ahora construimos una CNN real con Keras. La arquitectura sigue el patrón clásico:

```
Conv2D + ReLU  →  MaxPool  →  Conv2D + ReLU  →  MaxPool  →  Flatten  →  Dense  →  Softmax
  (extrae patrones locales)          (abstrae)              (clasifica)
```

In [ ]:
# Preprocesamiento para CNN: mantenemos la forma 2D (28, 28, 1)
# El último eje es el canal — 1 para escala de grises, 3 para RGB
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(f"Shape para CNN: {X_train_cnn.shape}")
print("(muestras, alto, ancho, canales)")

In [ ]:
# La CNN
cnn_mnist = keras.Sequential([
    # --- Bloque convolucional 1 ---
    # 32 filtros de 3x3. Entrada: (28,28,1) → Salida: (26,26,32)
    # 26 = 28 - 3 + 1 (la convolución 'válida' recorta los bordes)
    layers.Conv2D(32, kernel_size=3, activation='relu', input_shape=(28, 28, 1)),
    # Salida: (13,13,32) — mitad del tamaño
    layers.MaxPooling2D(pool_size=2),

    # --- Bloque convolucional 2 ---
    # 64 filtros. Aprende combinaciones de los patrones anteriores
    # Entrada: (13,13,32) → Salida: (11,11,64)
    layers.Conv2D(64, kernel_size=3, activation='relu'),
    # Salida: (5,5,64)
    layers.MaxPooling2D(pool_size=2),

    # --- Transición a clasificación ---
    # Flatten: (5,5,64) → (1600,) — aplanamos para la capa densa
    layers.Flatten(),

    # --- Clasificador ---
    # Capa densa: combina todas las features extraídas por las conv
    layers.Dense(128, activation='relu'),
    # Salida: 10 probabilidades (una por dígito)
    layers.Dense(10, activation='softmax')
], name='CNN_MNIST')

cnn_mnist.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_mnist.summary()

### Entendiendo el summary


| Capa | Output shape | ¿Qué pasó? |
|------|-------------|------------|
| Input | (28, 28, 1) | Imagen original, 1 canal |
| Conv2D(32) | (26, 26, 32) | 32 feature maps, ligeramente más pequeños |
| MaxPool | (13, 13, 32) | Mitad del tamaño |
| Conv2D(64) | (11, 11, 64) | 64 feature maps más abstractos |
| MaxPool | (5, 5, 64) | Mitad del tamaño |
| Flatten | (1600,) | Todo comprimido en un vector |
| Dense(128) | (128,) | **La representación final** |
| Dense(10) | (10,) | Una probabilidad por clase |


In [ ]:
# Entrenamos la CNN
hist_cnn = cnn_mnist.fit(
    X_train_cnn, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    verbose=0
)

loss_cnn, acc_cnn = cnn_mnist.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nResultados comparados — mismo dataset, mismas épocas:")
print(f"  MLP  → Accuracy: {acc_mlp:.4f} ({acc_mlp*100:.1f}%)  — {mlp_mnist.count_params():,} parámetros")
print(f"  CNN  → Accuracy: {acc_cnn:.4f} ({acc_cnn*100:.1f}%)  — {cnn_mnist.count_params():,} parámetros")

In [ ]:
# Curvas de entrenamiento comparadas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metrica, titulo in zip(
    axes,
    ['loss', 'accuracy'],
    ['Pérdida', 'Accuracy']
):
    ax.plot(hist_mlp.history[f'val_{metrica}'], label='MLP', color='#e63946', linewidth=2)
    ax.plot(hist_cnn.history[f'val_{metrica}'], label='CNN', color='#2a9d8f', linewidth=2)
    ax.set_title(f'{titulo} en validación', fontsize=13)
    ax.set_xlabel('Época')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('MLP vs CNN en MNIST — validación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bloque 4 — Visualización de filtros y feature maps 

### Parte A: Los filtros aprendidos en la primera capa

La primera capa Conv2D tiene 32 filtros de 3×3. Cada filtro es una pequeña imagen de 3×3 — los valores que la red ha aprendido durante el entrenamiento. Podemos visualizarlos directamente.

In [ ]:
# Extraemos los pesos de la primera capa convolucional
# Los pesos tienen shape: (3, 3, 1, 32) — (alto, ancho, canales_entrada, num_filtros)
pesos_capa1 = cnn_mnist.layers[0].get_weights()[0]
print(f"Shape de los pesos de la primera Conv2D: {pesos_capa1.shape}")
print("(alto_filtro, ancho_filtro, canales_entrada, num_filtros)")

# Visualizamos los 32 filtros
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    filtro = pesos_capa1[:, :, 0, i]  # El i-ésimo filtro
    # Normalizamos para visualización
    filtro_norm = (filtro - filtro.min()) / (filtro.max() - filtro.min() + 1e-8)
    ax.imshow(filtro_norm, cmap='RdBu_r', vmin=0, vmax=1)
    ax.axis('off')

plt.suptitle('Los 32 filtros aprendidos en la primera capa Conv2D\n'
             'Cada cuadro es un filtro 3×3 — los valores que la red aprendió durante el entrenamiento',
             fontsize=11)
plt.tight_layout()
plt.show()

print("\n🔍 ¿Reconoces detectores de bordes? ¿Esquinas? ¿Gradientes?")
print("   Compáralos con los filtros manuales que viste antes.")

### Parte B: Feature maps — qué 've' cada filtro en una imagen concreta

In [ ]:
# Construimos un modelo funcional para extraer activaciones intermedias
# (Keras 3 no expone .input/.output en modelos Sequential — usamos la API funcional)
inp = keras.Input(shape=(28, 28, 1))
x_pool1 = cnn_mnist.layers[0](inp)           # tras Conv2D(32)
x_pool1 = cnn_mnist.layers[1](x_pool1)       # tras MaxPool
x_conv2 = cnn_mnist.layers[2](x_pool1)       # tras Conv2D(64)
x_pool2 = cnn_mnist.layers[3](x_conv2)       # tras MaxPool
x_flat  = cnn_mnist.layers[4](x_pool2)       # tras Flatten
x_dense = cnn_mnist.layers[5](x_flat)        # tras Dense(128) ← representación

modelo_activaciones = Model(
    inputs=inp,
    outputs=[x_pool1, x_pool2, x_dense]
)

# Elegimos un dígito para inspeccionar — prueba a cambiar el índice
idx = 7
imagen = X_test_cnn[idx:idx+1]
clase_real = y_test[idx]

# Obtenemos las activaciones internas
fmaps_conv1, fmaps_conv2, repr_final = modelo_activaciones.predict(imagen, verbose=0)

print(f"Imagen: dígito '{clase_real}'")
print(f"Shape feature maps Conv1: {fmaps_conv1.shape}  — 32 mapas de 13×13")
print(f"Shape feature maps Conv2: {fmaps_conv2.shape}  — 64 mapas de 5×5")
print(f"Shape representación final: {repr_final.shape}  — vector de 128 dimensiones")

In [ ]:
# Visualizamos los feature maps de la primera capa
fig = plt.figure(figsize=(16, 7))
gs = gridspec.GridSpec(4, 9, figure=fig, hspace=0.3, wspace=0.1)

# Imagen original a la izquierda
ax_orig = fig.add_subplot(gs[:, 0])
ax_orig.imshow(X_test_raw[idx], cmap='gray')
ax_orig.set_title(f"'{clase_real}'", fontsize=14, fontweight='bold')
ax_orig.axis('off')

# Los 32 feature maps de Conv1
for i in range(32):
    row, col = divmod(i, 8)
    ax = fig.add_subplot(gs[row, col + 1])
    fmap = fmaps_conv1[0, :, :, i]
    ax.imshow(fmap, cmap='hot')
    ax.axis('off')

plt.suptitle('Feature maps de la primera capa Conv2D\n'
             'Cada mapa muestra dónde activó uno de los 32 filtros',
             fontsize=12, fontweight='bold')
plt.show()

print("🔍 Algunos filtros detectan bordes del dígito.")
print("   Algunos se activan en zonas específicas (arriba, abajo, curvas).")
print("   Algunos parecen 'apagados' — no detectaron ningún patrón relevante en este dígito.")

In [ ]:
# Feature maps de la segunda capa — ya menos interpretables
fig, axes = plt.subplots(8, 8, figsize=(14, 14))
for i, ax in enumerate(axes.flat):
    fmap = fmaps_conv2[0, :, :, i]
    ax.imshow(fmap, cmap='viridis')
    ax.axis('off')

plt.suptitle('Feature maps de la segunda capa Conv2D (64 filtros)\n'
             'Más abstractos: ya no reconocemos bordes simples, son combinaciones de patrones',
             fontsize=11)
plt.tight_layout()
plt.show()


- **Capa 1**: feature maps interpretables. Reconoces bordes, esquinas, gradientes. Son patrones locales simples.
- **Capa 2**: feature maps abstractos. Ya no puedes decir "ese detecta bordes horizontales". Son combinaciones de patrones de la capa anterior.

Si tuvieras una tercera capa, sería aún más abstracta: combinaciones de combinaciones.

Esta jerarquía de representaciones es la idea central del Deep Learning:  
**capas bajas = patrones simples, capas altas = conceptos abstractos**.

En una red entrenada para reconocer caras: capa 1 detecta bordes, capa 2 detecta ojos y narices, capa 3 detecta caras completas.

---
## Bloque 5 — El espacio de representaciones

Cuando la imagen pasa por todas las capas convolucionales y llega al `Flatten`, se convierte en un vector de 1600 dimensiones. La capa `Dense(128)` lo comprime a **128 dimensiones**.

Ese vector de 128 números es la **representación** que la red ha aprendido para esa imagen. Es todo lo que la red "sabe" sobre la imagen en ese punto.

**La pregunta importante**: ¿cómo están organizadas esas representaciones? ¿Imágenes del mismo dígito tienen representaciones parecidas?

Para verlo necesitamos reducir 128 dimensiones a 2 (para poder dibujarlas). Usaremos **t-SNE** — un algoritmo que intenta preservar las relaciones de vecindad al reducir dimensionalidad.

> **La siguiente celda tardará varios minutos.**

In [ ]:
# Extraemos las representaciones de 128 dimensiones para todo el test set
inp_repr = keras.Input(shape=(28, 28, 1))
x_r = inp_repr
for layer in cnn_mnist.layers[:-1]:  # Todas menos la última (softmax)
    x_r = layer(x_r)
modelo_repr = Model(inputs=inp_repr, outputs=x_r)

print("Extrayendo representaciones para 10.000 imágenes de test...")
representaciones = modelo_repr.predict(X_test_cnn, batch_size=256, verbose=1)
print(f"\nShape: {representaciones.shape}")
print("10.000 imágenes × 128 dimensiones")

In [ ]:
# t-SNE: reducimos de 128 a 2 dimensiones
# Esta celda tarda varios minutos — es normal
print("Ejecutando t-SNE (128D → 2D)...")
print("Esto puede tardar entre 3 y 10 minutos. El algoritmo está calculando")
print("similitudes entre los 10.000 vectores de representación.")
print()

tsne = TSNE(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=42,
    verbose=1
)
repr_2d = tsne.fit_transform(representaciones)
print(f"\n✅ t-SNE completado. Shape resultante: {repr_2d.shape}")

In [ ]:
# Visualizamos el espacio de representaciones
colores = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(12, 10))

for digito in range(10):
    mask = y_test == digito
    ax.scatter(
        repr_2d[mask, 0],
        repr_2d[mask, 1],
        c=[colores[digito]],
        label=str(digito),
        alpha=0.4,
        s=8
    )

# Añadimos etiquetas en el centroide de cada cluster
for digito in range(10):
    mask = y_test == digito
    cx, cy = repr_2d[mask, 0].mean(), repr_2d[mask, 1].mean()
    ax.text(cx, cy, str(digito), fontsize=20, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_title('Espacio de representaciones de la CNN (t-SNE: 128D → 2D)\n'
             'Cada punto es una imagen de test. El color indica el dígito real.',
             fontsize=13, fontweight='bold')
ax.legend(title='Dígito', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

print("\n🔍 Observaciones:")
print("   - Cada dígito forma un cluster separado — la red los ha aprendido a separar")
print("   - Dígitos visualmente similares (4 y 9, 3 y 8) están más cerca")
print("   - La red nunca vio este mapa — emergió sola del entrenamiento")

La red ha aprendido a **organizar el espacio** de tal forma que imágenes similares quedan cerca y clases distintas quedan separadas. Nadie le dijo "pon los 3s aquí y los 7s allá" — emergió del entrenamiento.

Ese vector de 128 dimensiones **es** lo que la red sabe del input. Contiene toda la información relevante para la clasificación, comprimida desde 784 píxeles.

Este concepto tiene un nombre: **embedding**. Y aparece en todas partes:

| Modelo | Entrada | Embedding |
|--------|---------|----------|
| CNN de imágenes | Píxeles (784 dims) | Vector 128D que representa "qué hay en la imagen" |
| Autoencoder | Imagen | Vector comprimido desde el que se puede reconstruir la imagen |
| GAN | Ruido aleatorio | El generador aprende a mapear ruido → imagen realista pasando por el mismo espacio |
| LLM | Tokens de texto | Vector que representa "qué significa esta palabra en este contexto" |

La idea de fondo es siempre la misma: **comprimir la entrada en una representación densa donde las relaciones semánticas se convierten en geometría**.

---
## Bloque 6 — CIFAR-10: arquitectura en tus manos 

MNIST es demasiado fácil para ver las diferencias entre arquitecturas. CIFAR-10 tiene 60.000 imágenes en color de 32×32 píxeles en 10 categorías: aviones, coches, pájaros, gatos, ciervos, perros, ranas, caballos, barcos y camiones.

Aquí la tarea es genuinamente difícil. Un humano acierta >95%. Los modelos del estado del arte llegan a ~99%. Nosotros vamos a ver cuánto conseguimos con una CNN pequeña, y cómo cambia al modificar la arquitectura.

In [ ]:
# Cargamos CIFAR-10 — ~170MB, tarda un momento en descargarse
(Xc_train, yc_train), (Xc_test, yc_test) = keras.datasets.cifar10.load_data()

# Normalizamos
Xc_train = Xc_train.astype('float32') / 255.0
Xc_test  = Xc_test.astype('float32') / 255.0

# Las etiquetas vienen con shape (N, 1) — las aplanamos
yc_train = yc_train.flatten()
yc_test  = yc_test.flatten()

CLASES = ['avión', 'coche', 'pájaro', 'gato', 'ciervo',
          'perro', 'rana', 'caballo', 'barco', 'camión']

print(f"CIFAR-10 train: {Xc_train.shape}")
print(f"CIFAR-10 test:  {Xc_test.shape}")
print(f"Clases: {CLASES}")

# Visualizamos algunos ejemplos
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(Xc_train[i])
    ax.set_title(CLASES[yc_train[i]], fontsize=7)
    ax.axis('off')
plt.suptitle('Primeros 20 ejemplos de CIFAR-10', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# CNN base para CIFAR-10 — intencionadamente modesta
# Tu tarea: modificarla para mejorar el accuracy en validación

def construir_cnn_cifar(config='base'):
    """
    config='base'    → La arquitectura inicial que te damos
    config='tuya'    → Modifica las capas marcadas con ### MODIFICA AQUÍ
    """
    model = keras.Sequential(name=f'CNN_CIFAR_{config}')

    if config == 'base':
        model.add(layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)))
        model.add(layers.MaxPooling2D(2))
        model.add(layers.Conv2D(64, 3, activation='relu'))
        model.add(layers.MaxPooling2D(2))
        model.add(layers.Flatten())
        model.add(layers.Dense(128, activation='relu'))
        model.add(layers.Dense(10, activation='softmax'))

    elif config == 'tuya':
        ### MODIFICA AQUÍ — algunas ideas:
        ### - Añadir un tercer bloque Conv2D + MaxPool
        ### - Aumentar el número de filtros (64 → 128 en la segunda capa)
        model.add(layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)))
        model.add(layers.MaxPooling2D(2))
        model.add(layers.Conv2D(64, 3, activation='relu'))
        model.add(layers.MaxPooling2D(2))
        model.add(layers.Flatten())
        model.add(layers.Dense(128, activation='relu'))
        model.add(layers.Dense(10, activation='softmax'))

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_cifar_base = construir_cnn_cifar('base')
cnn_cifar_base.summary()

In [ ]:
# Entrenamos la arquitectura base — 15 épocas para ver si hay margen de mejora
hist_cifar_base = cnn_cifar_base.fit(
    Xc_train, yc_train,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    verbose=0
)

_, acc_cifar_base = cnn_cifar_base.evaluate(Xc_test, yc_test, verbose=0)
print(f"CNN base — Accuracy en test: {acc_cifar_base:.4f} ({acc_cifar_base*100:.1f}%)")
print("\nAhora modifica la arquitectura en la celda anterior (config='tuya') e intenta superarlo.")

In [ ]:
# --- CELDA DE EXPERIMENTACIÓN ---
# Entrena tu arquitectura modificada y compara

cnn_cifar_tuya = construir_cnn_cifar('tuya')

hist_cifar_tuya = cnn_cifar_tuya.fit(
    Xc_train, yc_train,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    verbose=0
)

_, acc_cifar_tuya = cnn_cifar_tuya.evaluate(Xc_test, yc_test, verbose=0)

print("=" * 45)
print("COMPARACIÓN")
print("=" * 45)
print(f"  Base: {acc_cifar_base:.4f} ({acc_cifar_base*100:.1f}%) — {cnn_cifar_base.count_params():,} params")
print(f"  Tuya: {acc_cifar_tuya:.4f} ({acc_cifar_tuya*100:.1f}%) — {cnn_cifar_tuya.count_params():,} params")

# Curvas comparadas
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_cifar_base.history['val_accuracy'], label='Base', color='#e63946', linewidth=2)
ax.plot(hist_cifar_tuya.history['val_accuracy'], label='Tuya', color='#2a9d8f', linewidth=2)
ax.set_title('Accuracy en validación — CIFAR-10', fontsize=13)
ax.set_xlabel('Época')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

###  Preguntas de reflexión

**1. Añadiste más capas y el accuracy bajó. ¿Por qué puede pasar?**

<details>
<summary>💡 Respuesta</summary>

Con CIFAR-10 (32×32 píxeles) cada MaxPool reduce el tamaño a la mitad. Si pones demasiados bloques Conv+Pool, las feature maps se vuelven tan pequeñas (2×2, 1×1) que ya no hay información espacial que extraer. También puede ser overfitting: más capacidad sin regularización memorizaen lugar de generalizar. Antes de añadir capas, añade Dropout o BatchNormalization.

</details>


</details>

**2. La CNN base llega a ~70% en CIFAR-10 pero a ~99% en MNIST. ¿Por qué tanta diferencia?**

<details>
<summary>💡 Respuesta</summary>

MNIST tiene dígitos centrados, fondo negro uniforme, sin variación de iluminación ni rotación. Las clases son muy distintas entre sí. CIFAR-10 tiene imágenes reales: un perro puede estar en cualquier posición, con cualquier iluminación, parcialmente tapado, de razas distintas. Los píxeles de un gato y un perro son mucho más similares entre sí que los de un '3' y un '7'. Para llegar a >90% en CIFAR-10 necesitas arquitecturas más profundas

</details>